In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

print("Libraries imported successfully!")

Libraries imported successfully!


In [16]:
# Upload dataset

uploaded = files.upload()

# Load CSV file
df = pd.read_csv("retail_sales_raw_dataset.csv")

print("Dataset loaded successfully!")

Saving retail_sales_raw_dataset.csv to retail_sales_raw_dataset (3).csv
Dataset loaded successfully!


In [18]:
# Creating a working copy

df_clean = df.copy()

print("Raw dataset:", df.shape)
print("Working dataset:", df_clean.shape)

Raw dataset: (5020, 17)
Working dataset: (5020, 17)


In [19]:
# Count duplicate records

duplicate_count = df_clean.duplicated().sum()

print("Duplicate records found:", duplicate_count)

Duplicate records found: 20


In [20]:
# Remove duplicate records

df_clean = df_clean.drop_duplicates()

print("Duplicates remaining:", df_clean.duplicated().sum())
print("New dataset shape:", df_clean.shape)

Duplicates remaining: 0
New dataset shape: (5000, 17)


In [21]:
# Missing values report

missing_report = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (
        df_clean.isnull().mean() * 100
    ).round(2)
})

missing_report[missing_report["Missing_Count"] > 0]

,Missing_Count,Missing_Percentage
age,90,1.8
discount_pct,60,1.2
shipping_days,75,1.5


In [22]:
# Age

df_clean["age"] = df_clean["age"].fillna(
    df_clean["age"].median()
)

In [23]:
# Discount percentage

df_clean["discount_pct"] = df_clean["discount_pct"].fillna(0)

In [24]:
# Shipping days

df_clean["shipping_days"] = df_clean["shipping_days"].fillna(
    df_clean["shipping_days"].median()
)

In [25]:
print(df_clean.isnull().sum())

order_line_id     0
order_date        0
customer_id       0
gender            0
age               0
city              0
state             0
product           0
category          0
quantity          0
unit_price_ngn    0
unit_cost_ngn     0
discount_pct      0
payment_method    0
sales_channel     0
shipping_days     0
order_status      0
dtype: int64


In [26]:
# Standardise categorical text columns

text_columns = [
    "gender",
    "city",
    "state",
    "product",
    "category",
    "payment_method",
    "sales_channel",
    "order_status"
]

for column in text_columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
        .str.title()
    )

print("Text standardisation completed!")

Text standardisation completed!


In [27]:
print("Categories:")
print(df_clean["category"].unique())

print("\nPayment methods:")
print(df_clean["payment_method"].unique())

Categories:
<StringArray>
['Fashion', 'Electronics', 'Beauty', 'Groceries', 'Home & Kitchen']
Length: 5, dtype: string

Payment methods:
<StringArray>
['Wallet', 'Card', 'Ussd', 'Cash On Delivery', 'Bank Transfer']
Length: 5, dtype: string


In [28]:
# Convert order_date into datetime format

df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    errors="coerce"
)

print("Order date converted to datetime!")

Order date converted to datetime!


In [31]:
# Convert numerical columns

df_clean["quantity"] = pd.to_numeric(
    df_clean["quantity"], errors="coerce"
)

df_clean["unit_price_ngn"] = pd.to_numeric(
    df_clean["unit_price_ngn"], errors="coerce"
)

df_clean["unit_cost_ngn"] = pd.to_numeric(
    df_clean["unit_cost_ngn"], errors="coerce"
)

df_clean["discount_pct"] = pd.to_numeric(
    df_clean["discount_pct"], errors="coerce"
)

In [32]:
# Convert whole-number columns after filling missing values

df_clean["age"] = df_clean["age"].astype(int)
df_clean["shipping_days"] = df_clean["shipping_days"].astype(int)
df_clean["quantity"] = df_clean["quantity"].astype(int)

print(df_clean.dtypes)

order_line_id             object
order_date        datetime64[ns]
customer_id               object
gender            string[python]
age                        int64
city              string[python]
state             string[python]
product           string[python]
category          string[python]
quantity                   int64
unit_price_ngn             int64
unit_cost_ngn              int64
discount_pct             float64
payment_method    string[python]
sales_channel     string[python]
shipping_days              int64
order_status      string[python]
dtype: object


In [33]:
# Check invalid values

print("Invalid quantity:")
print((df_clean["quantity"] <= 0).sum())

print("Invalid selling price:")
print((df_clean["unit_price_ngn"] < 0).sum())

print("Invalid unit cost:")
print((df_clean["unit_cost_ngn"] < 0).sum())

print("Invalid discount:")
print(
    (~df_clean["discount_pct"].between(0, 1)).sum()
)

print("Invalid age:")
print(
    (~df_clean["age"].between(18, 70)).sum()
)

print("Missing order dates:")
print(df_clean["order_date"].isnull().sum())

Invalid quantity:
0
Invalid selling price:
0
Invalid unit cost:
0
Invalid discount:
0
Invalid age:
0
Missing order dates:
0


## Feature Engineering
1. Gross Sales
2. Discount
3. Net Sales
4. Total cost
5. Profit

In [34]:
# Calculate gross sales

df_clean["gross_sales"] = (
    df_clean["quantity"] *
    df_clean["unit_price_ngn"]
)

# Calculate discount amount

df_clean["discount_amount"] = (
    df_clean["gross_sales"] *
    df_clean["discount_pct"]
)

# Calculate net sales

df_clean["net_sales"] = (
    df_clean["gross_sales"] -
    df_clean["discount_amount"]
)

# Calculate total cost

df_clean["total_cost"] = (
    df_clean["quantity"] *
    df_clean["unit_cost_ngn"]
)

# Calculate profit

df_clean["profit"] = (
    df_clean["net_sales"] -
    df_clean["total_cost"]
)

# Calculate profit margin

df_clean["profit_margin_pct"] = np.where(
    df_clean["net_sales"] != 0,
    (df_clean["profit"] / df_clean["net_sales"]) * 100,
    np.nan
)

print("Feature engineering completed!")

Feature engineering completed!


In [37]:
df_clean[
    [
        "product",
        "quantity",
        "gross_sales",
        "discount_amount",
        "net_sales",
        "total_cost",
        "profit",
        "profit_margin_pct"
    ]
].head(10)

,product,quantity,gross_sales,discount_amount,net_sales,total_cost,profit,profit_margin_pct
0,T-Shirt,2,33200,3320.0,29880.0,18400,11480.0,38.420348
1,Smartphone,1,200200,20020.0,180180.0,135800,44380.0,24.630925
2,Makeup Kit,5,160500,8025.0,152475.0,69500,82975.0,54.418757
3,Rice 10Kg,1,23100,0.0,23100.0,19700,3400.0,14.718615
4,Smartphone,2,371400,37140.0,334260.0,269000,65260.0,19.523724
5,Handbag,1,46200,4620.0,41580.0,25500,16080.0,38.672439
6,Bedsheet Set,1,39400,3940.0,35460.0,20200,15260.0,43.034405
7,Cooking Oil 5L,1,18400,1840.0,16560.0,13400,3160.0,19.082126
8,Laptop,3,1984500,99225.0,1885275.0,1482900,402375.0,21.343040
9,Smartphone,2,346800,17340.0,329460.0,282400,47060.0,14.283980


In [38]:
# Analyse monthly and yearly sales trends

df_clean["year"] = df_clean["order_date"].dt.year

df_clean["month"] = df_clean["order_date"].dt.month

df_clean["month_name"] = df_clean["order_date"].dt.month_name()

df_clean["quarter"] = (
    "Q" + df_clean["order_date"].dt.quarter.astype(str)
)

print("Date features created!")

Date features created!


In [39]:
# Final cleaning checks

print("FINAL DATA QUALITY REPORT")
print("--------------------------")

print("Total rows:", len(df_clean))
print("Total columns:", len(df_clean.columns))

print("Duplicate rows:", df_clean.duplicated().sum())

print("Missing values:", df_clean.isnull().sum().sum())

print("\nData types:")
print(df_clean.dtypes)

FINAL DATA QUALITY REPORT
--------------------------
Total rows: 5000
Total columns: 27
Duplicate rows: 0
Missing values: 0

Data types:
order_line_id                object
order_date           datetime64[ns]
customer_id                  object
gender               string[python]
age                           int64
city                 string[python]
state                string[python]
product              string[python]
category             string[python]
quantity                      int64
unit_price_ngn                int64
unit_cost_ngn                 int64
discount_pct                float64
payment_method       string[python]
sales_channel        string[python]
shipping_days                 int64
order_status         string[python]
gross_sales                   int64
discount_amount             float64
net_sales                   float64
total_cost                    int64
profit                      float64
profit_margin_pct           float64
year                          int32

In [48]:
# Export to CSV

df_clean.to_csv(
    "retail_sales_cleaned.csv",
    index=False
)

print("Cleaned CSV exported successfully!")

Cleaned CSV exported successfully!


In [49]:
files.download("retail_sales_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
# Export to Excel

df_clean.to_excel(
    "retail_sales_cleaned.xlsx",
    index=False
)

print("Cleaned Excel file exported successfully!")

Cleaned Excel file exported successfully!


In [52]:
files.download("retail_sales_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>